# Trusted Zone
Secuencia de la fase:
1. Configuracion de Spark, rutas y acceso a formatted.
2. Carga de tablas formateadas desde DuckDB.
3. Aplicacion de reglas de calidad a POIs, hoteles y meteo.
4. Aplicacion de reglas de calidad a tablas Airbnb y reconciliacion del calendario.
5. Persistencia de tablas limpias en trusted y reporte de calidad.


In [1]:
import json
import shutil
import tempfile
from pathlib import Path

import duckdb
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F

ROOT = Path.cwd()
FORMATTED_DB = ROOT / "formatted_zone" / "formatted_zone.duckdb"
TRUSTED_ZONE = ROOT / "trusted_zone"
TRUSTED_DB = TRUSTED_ZONE / "trusted_zone.duckdb"
TRUSTED_REPORTS = TRUSTED_ZONE / "reports"
DATASETS = {
    "pics": {"path": ROOT / "opendatabcn_pics-csv.csv", "encoding": "utf-16"},
    "weather": {"path": ROOT / "dataset_prat_temps_2025-26.csv", "encoding": "utf-8-sig"},
    "hotels": {"path": ROOT / "opendatabcn_allotjament_hotels-csv.csv", "encoding": "utf-16"},
    "airbnb_listings": {"path": ROOT / "listings.csv", "encoding": "utf-8-sig"},
    "airbnb_neighbourhoods": {"path": ROOT / "neighbourhoods.csv", "encoding": "utf-8-sig"},
    "airbnb_reviews": {"path": ROOT / "reviews_airbnb.csv", "encoding": "utf-8-sig"},
    "airbnb_calendar": {"path": ROOT / "calendar.csv", "encoding": "utf-8-sig"},
    "income_2022": {"path": ROOT / "2022_renda_disponible_llars_per_persona.csv", "encoding": "utf-8-sig"},
    "hut_licenses": {"path": ROOT / "hut_comunicacio_opendata.csv", "encoding": "utf-8-sig"},
}

CORE_ZONE_TABLES = [
    "pics",
    "hotels",
    "weather",
    "airbnb_listings",
    "airbnb_neighbourhoods",
    "airbnb_reviews",
    "income_2022",
    "hut_licenses",
]


def ensure_phase_dirs() -> None:
    for path in [TRUSTED_REPORTS]:
        path.mkdir(parents=True, exist_ok=True)


In [2]:
def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def temp_dir(prefix: str) -> Path:
    return Path(tempfile.mkdtemp(prefix=prefix))

def remove_path(path: Path) -> None:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    else:
        path.unlink(missing_ok=True)

def write_duckdb_table_from_parquet_glob(db_path: Path, table_name: str, parquet_glob: str) -> None:
    if duckdb is None:
        raise RuntimeError("DuckDB no está instalado.")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = duckdb.connect(str(db_path))
    try:
        conn.execute(f'DROP TABLE IF EXISTS "{table_name}"')
        conn.execute(
            f"""
            CREATE TABLE "{table_name}" AS
            SELECT *
            FROM read_parquet('{parquet_glob}')
            """
        )
        conn.commit()
    finally:
        conn.close()


def resolve_input_path(dataset_name: str) -> Path:
    return DATASETS[dataset_name]["path"]


In [3]:
def spark_from_duckdb_table(spark: SparkSession, table_name: str) -> DataFrame:
    parquet_dir = temp_dir(f"formatted_{table_name}_")
    parquet_file = parquet_dir / f"{table_name}.parquet"
    import duckdb

    conn = duckdb.connect(str(FORMATTED_DB), read_only=True)
    try:
        conn.execute(f"COPY (SELECT * FROM \"{table_name}\") TO '{parquet_file.as_posix()}' (FORMAT PARQUET)")
    finally:
        conn.close()
    dataframe = spark.read.parquet(str(parquet_file)).cache()
    dataframe.count()
    remove_path(parquet_dir)
    return dataframe

def in_barcelona() -> F.Column:
    return (
        F.col("latitude").isNotNull()
        & F.col("longitude").isNotNull()
        & F.col("latitude").between(41.2, 41.5)
        & F.col("longitude").between(2.0, 2.3)
    )

def deduplicate(dataframe: DataFrame, partition_columns: list[str]) -> DataFrame:
    window = Window.partitionBy(*partition_columns).orderBy(*partition_columns)
    return dataframe.withColumn("_rownum", F.row_number().over(window)).where(F.col("_rownum") == 1).drop("_rownum")

def quality_metrics(original: DataFrame, cleaned: DataFrame, dataset: str, rules: dict[str, int]) -> dict[str, object]:
    original_rows = original.count()
    valid_rows = cleaned.count()
    return {
        "dataset": dataset,
        "original_rows": original_rows,
        "valid_rows": valid_rows,
        "removed_rows": original_rows - valid_rows,
        "rules": rules,
    }

def persist_trusted_table(dataframe: DataFrame, table_name: str) -> int:
    parquet_dir = temp_dir(f"trusted_{table_name}_")
    try:
        dataframe.write.mode("overwrite").parquet(str(parquet_dir))
        write_duckdb_table_from_parquet_glob(TRUSTED_DB, table_name, f"{parquet_dir.as_posix()}/*.parquet")
        return dataframe.count()
    finally:
        remove_path(parquet_dir)


In [4]:
ensure_phase_dirs()
if SparkSession is None:
    raise RuntimeError("pyspark no está instalado. La trusted zone requiere Spark.")
spark = SparkSession.builder.appName("bda-trusted-zone-notebook").master("local[*]").getOrCreate()

pics = spark_from_duckdb_table(spark, "pics")
pics_clean = deduplicate(
    pics.where(F.col("record_id").isNotNull() & F.col("name").isNotNull() & F.col("district_name").isNotNull() & in_barcelona()),
    ["record_id", "attribute_name", "attribute_value"],
)

hotels = spark_from_duckdb_table(spark, "hotels")
hotels_clean = deduplicate(
    hotels.where(F.col("record_id").isNotNull() & F.col("name").isNotNull() & F.col("district_name").isNotNull() & in_barcelona()),
    ["record_id"],
)

weather = spark_from_duckdb_table(spark, "weather")
weather_clean = deduplicate(
    weather.where(
        F.col("timestamp").isNotNull()
        & F.col("temperature_avg").between(-20, 50)
        & F.col("humidity_avg").between(0, 100)
        & F.col("pressure_avg").between(850, 1100)
        & (F.col("radiation_avg") >= 0)
        & (F.col("rain_total") >= 0)
    ),
    ["timestamp"],
)

neighbourhoods = spark_from_duckdb_table(spark, "airbnb_neighbourhoods")
neighbourhoods_clean = deduplicate(
    neighbourhoods.where(F.col("district_name").isNotNull() & F.col("neighborhood_name").isNotNull()),
    ["district_name", "neighborhood_name"],
)

reviews = spark_from_duckdb_table(spark, "airbnb_reviews")
reviews_clean = deduplicate(reviews.where(F.col("listing_id").isNotNull()), ["listing_id"])


income_rows = spark_from_duckdb_table(spark, "income_2022")
income_clean = deduplicate(
    income_rows.where(
        F.col("year").isNotNull()
        & F.col("district_name").isNotNull()
        & F.col("neighborhood_name").isNotNull()
        & F.col("income_eur").isNotNull()
        & (F.col("income_eur") > 0)
    ),
    ["year", "district_id", "neighborhood_id", "census_section"],
)

hut_rows = spark_from_duckdb_table(spark, "hut_licenses")
hut_clean = deduplicate(
    hut_rows.where(
        F.col("license_id").isNotNull()
        & F.col("district_name").isNotNull()
        & F.col("neighborhood_name").isNotNull()
        & F.col("latitude").between(41.2, 41.5)
        & F.col("longitude").between(2.0, 2.3)
    ),
    ["license_id"],
)


In [5]:
neighbourhood_map = neighbourhoods_clean.select(
    F.col("neighborhood_name"),
    F.col("district_name").alias("district_name_lookup"),
).dropDuplicates(["neighborhood_name"])

listings = spark_from_duckdb_table(spark, "airbnb_listings")
listings_enriched = listings.join(neighbourhood_map, on="neighborhood_name", how="left")
listings_enriched = listings_enriched.withColumn(
    "district_name",
    F.coalesce(F.col("district_name"), F.col("district_name_lookup")),
).drop("district_name_lookup")
listings_clean = deduplicate(
    listings_enriched.where(
        F.col("listing_id").isNotNull()
        & F.col("neighborhood_name").isNotNull()
        & F.col("district_name").isNotNull()
        & F.col("latitude").between(41.2, 41.5)
        & F.col("longitude").between(2.0, 2.3)
        & F.col("price").isNotNull()
        & (F.col("price") > 0)
        & F.col("accommodates").isNotNull()
        & (F.col("accommodates") > 0)
        & ((F.col("review_scores_rating").isNull()) | F.col("review_scores_rating").between(0, 5))
    ),
    ["listing_id"],
)

valid_listing_ids = listings_clean.select("listing_id").dropDuplicates(["listing_id"])
calendar_encoding = DATASETS["airbnb_calendar"]["encoding"]
if calendar_encoding == "utf-8-sig":
    calendar_encoding = "utf-8"
calendar_raw = spark.read.option("header", True).option("encoding", calendar_encoding).csv(str(resolve_input_path("airbnb_calendar")))
calendar_clean = (
    calendar_raw.select(
        F.lit("airbnb_calendar").alias("dataset"),
        F.trim(F.col("listing_id")).alias("listing_id"),
        F.trim(F.col("date")).alias("date"),
        F.trim(F.col("available")).alias("available"),
        F.trim(F.col("price")).alias("price"),
        F.trim(F.col("adjusted_price")).alias("adjusted_price"),
        F.trim(F.col("minimum_nights")).alias("minimum_nights"),
        F.trim(F.col("maximum_nights")).alias("maximum_nights"),
    )
    .join(valid_listing_ids, on="listing_id", how="inner")
    .dropDuplicates(["listing_id", "date"])
)

trusted_tables = {
    "pics": pics_clean,
    "hotels": hotels_clean,
    "weather": weather_clean,
    "airbnb_neighbourhoods": neighbourhoods_clean,
    "airbnb_reviews": reviews_clean,
    "airbnb_listings": listings_clean,
    "airbnb_calendar": calendar_clean,
    "income_2022": income_clean,
    "hut_licenses": hut_clean,
}
trusted_counts = {name: persist_trusted_table(df, name) for name, df in trusted_tables.items()}
quality_report = [
    quality_metrics(pics, pics_clean, "pics", {}),
    quality_metrics(hotels, hotels_clean, "hotels", {}),
    quality_metrics(weather, weather_clean, "weather", {}),
    quality_metrics(neighbourhoods, neighbourhoods_clean, "airbnb_neighbourhoods", {}),
    quality_metrics(reviews, reviews_clean, "airbnb_reviews", {}),
    quality_metrics(listings, listings_clean, "airbnb_listings", {}),
    quality_metrics(calendar_raw, calendar_clean, "airbnb_calendar", {}),
    quality_metrics(income_rows, income_clean, "income_2022", {}),
    quality_metrics(hut_rows, hut_clean, "hut_licenses", {}),
]
spark.stop()
write_json(TRUSTED_REPORTS / "trusted_zone_quality_report.json", quality_report)
trusted_counts


{'pics': 878, 'hotels': 446, 'weather': 20967, 'airbnb_neighbourhoods': 73, 'airbnb_reviews': 14421, 'airbnb_listings': 15276, 'airbnb_calendar': 5575744, 'income_2022': 1068, 'hut_licenses': 10724}